# CapCap server trên Google Colab + Tailscale

Notebook này chạy FastAPI và frontend React trực tiếp trong Colab. Video đi qua tailnet riêng, không dùng Cloudflare Quick Tunnel.

Trước khi chạy: build frontend bằng `npm run build`, đặt `TAILSCALE_AUTH_KEY` và credential Ollama trong Colab Secrets.

In [ ]:
# Chạy từ root của repo CapCap. Nếu repo chưa có trong Colab, upload/clone repo trước.
%pip -q install -r requirements-colab.txt
!apt-get update -qq && apt-get install -y -qq zstd ffmpeg
!curl -fsSL https://ollama.com/install.sh | sh
!curl -fsSL https://tailscale.com/install.sh | sh
!ollama --version
!tailscale version

In [ ]:
import os
import secrets
import subprocess
import sys
import time
from pathlib import Path

MODEL = os.getenv("OPENAI_MODEL", "gemma4:31b-cloud")
CAPCAP_PORT = 8765
os.environ["CAPCAP_APP_TOKEN"] = secrets.token_urlsafe(32)
os.environ["CAPCAP_COLAB_HOST"] = "127.0.0.1"
os.environ["CAPCAP_COLAB_PORT"] = str(CAPCAP_PORT)
os.environ["CAPCAP_COLAB_WORK_ROOT"] = "/content/capcap"
os.environ["CAPCAP_DRIVE_ROOT"] = "/content/drive/MyDrive/CapCap"
os.environ.setdefault("CAPCAP_MODEL_CACHE_MODE", "content")
os.environ["OLLAMA_MODELS"] = "/content/ollama-models"
os.environ["CAPCAP_FFMPEG_PATH"] = "/usr/bin/ffmpeg"
os.environ["CAPCAP_FFPROBE_PATH"] = "/usr/bin/ffprobe"

try:
    from google.colab import userdata
    tailscale_key = userdata.get("TAILSCALE_AUTH_KEY")
    ollama_key = userdata.get("OLLAMA_API_KEY")
    if tailscale_key:
        os.environ["TAILSCALE_AUTH_KEY"] = tailscale_key
    if ollama_key:
        os.environ["OLLAMA_API_KEY"] = ollama_key
except Exception:
    tailscale_key = os.getenv("TAILSCALE_AUTH_KEY", "")

if not os.environ.get("TAILSCALE_AUTH_KEY"):
    raise RuntimeError("Thiếu Colab Secret TAILSCALE_AUTH_KEY.")

drive_root = Path("/content/drive/MyDrive/CapCap")
drive_root.mkdir(parents=True, exist_ok=True)
if not Path("frontend/dist/index.html").exists():
    raise RuntimeError("Thiếu frontend/dist/index.html. Hãy chạy npm run build trước khi mở notebook.")

ollama_log = open("/content/ollama.log", "w")
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(45):
    try:
        import urllib.request
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(Path("/content/ollama.log").read_text(errors="replace"))

pull = subprocess.run(["ollama", "pull", MODEL], text=True, capture_output=True)
print(pull.stdout)
if pull.returncode != 0:
    print(pull.stderr)
    raise RuntimeError("Không pull được model. Nếu model cloud yêu cầu xác thực, chạy `!ollama signin` rồi chạy lại cell.")
print(f"Model ready: {MODEL}")

In [ ]:
# Tailscale userspace mode phù hợp với Colab vì không phụ thuộc /dev/net/tun.
tailscaled_log = open("/content/tailscaled.log", "w")
tailscaled_process = subprocess.Popen(["tailscaled", "--tun=userspace-networking", "--socks5-server=localhost:1055"], stdout=tailscaled_log, stderr=subprocess.STDOUT, start_new_session=True)
time.sleep(2)
up = subprocess.run(["tailscale", "up", "--authkey=" + os.environ["TAILSCALE_AUTH_KEY"], "--hostname=capcap-colab", "--accept-dns=true"], text=True, capture_output=True)
if up.returncode != 0:
    print(up.stdout, up.stderr)
    raise RuntimeError("Tailscale không khởi động được. Kiểm tra auth key và /content/tailscaled.log.")
print(subprocess.run(["tailscale", "status"], text=True, capture_output=True).stdout)

In [ ]:
# Start CapCap API ở localhost, sau đó publish qua Tailscale Serve.
server_log = open("/content/capcap-server.log", "w")
server_process = subprocess.Popen([sys.executable, "-m", "app.colab_server"], stdout=server_log, stderr=subprocess.STDOUT, start_new_session=True, env=os.environ.copy())
time.sleep(3)
health = subprocess.run([sys.executable, "-c", "import urllib.request; print(urllib.request.urlopen('http://127.0.0.1:8765/health').read().decode())"], text=True, capture_output=True)
print(health.stdout)
if health.returncode != 0:
    print(Path("/content/capcap-server.log").read_text(errors="replace"))
    raise RuntimeError("CapCap server chưa sẵn sàng.")
serve = subprocess.run(["tailscale", "serve", "--bg", "http://127.0.0.1:8765"], text=True, capture_output=True)
print(serve.stdout or serve.stderr)
print("=== CAPCAP SESSION ===")
print("Mở địa chỉ HTTPS được in bởi `tailscale serve status` trên máy Windows đã đăng nhập cùng tailnet.")
print("App token:", os.environ["CAPCAP_APP_TOKEN"])
print("Model:", MODEL)
print("======================")
print(subprocess.run(["tailscale", "serve", "status"], text=True, capture_output=True).stdout)

## Dừng phiên

Dừng Colab runtime sẽ dừng server, Tailscale node và làm token hết hiệu lực. Project metadata đã được mirror vào Google Drive; file nặng vẫn nằm trong `/content`.